<a href="https://colab.research.google.com/github/labphonlab/phonetics-research-statistics-support/blob/main/notebooks/ch16_reporting_reproducibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このバッジから開くと、設定なしでそのまま実行できます。
書き換えを残したいときは **ファイル → ドライブにコピーを保存** を一度行ってください。


# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第16回：手法の選び方・結果の報告・再現性


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本ファイルの公開用URLを、みだりに他の人へ共有しないこと。

本ファイルの公開用URLにアクセスすると、お手元のGoogleアカウントに本ファイルのコピーが作成される。
そのコピーはあなただけのファイルとして生成されるため、自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Meteyard & Davies (2020) 混合効果モデルの報告に関する調査／Simmons, Nelson & Simonsohn (2011) 研究者の自由度／Nosek et al. 事前登録。

In [ ]:
install.packages(c("lme4", "lmerTest"), repos = "https://cloud.r-project.org")
library(lme4)
library(lmerTest)
library(tidyverse)


In [ ]:
# ---- 共通データセットの読み込み ----
# 第1回と全く同じ手順・同じ乱数seedでVOTデータを作り直す。
# 「本ごとに違うデータ」を避けるためだが、外部URLから読み込むのではなく
# その場で再生成する設計にしてある——理由は、配布用ノートブックが
# 非公開のGitHubリポジトリのURLに依存すると、読者の環境からは
# アクセスできず404になってしまうため（実際にこの問題が起きて修正した）。
# コードを自分の環境で動かす分には、この方式のほうが「ネットワーク不要で
# 必ず同じ結果になる」という利点もある。
if (!requireNamespace("tidyverse", quietly = TRUE)) install.packages("tidyverse", repos = "https://cloud.r-project.org")
library(tidyverse)

set.seed(42)  # 乱数のseedを固定する。同じ数字なら誰が実行しても同じ結果になる
n_speaker <- 10                    # 話者数
n_word    <- 20                    # 単語数（各話者が全単語を1回ずつ発話する）
n_obs     <- n_speaker * n_word    # = 200行

# 何行目がどの話者・どの単語かを表す添字。あとで話者ごと・単語ごとの
# 値を引くのに使う（speaker_intercept[speaker_idx] のように書ける）。
speaker_idx <- rep(1:n_speaker, each  = n_word)
word_idx    <- rep(1:n_word,    times = n_speaker)

# 1行ごとにvoiced/voicelessを交互に並べる。話者1人あたり20行なので、
# どの話者を見ても必ずvoiced10回・voiceless10回が含まれる
# ——これは第5回で「話者ごとにconditionの効果を推定する」ために必須の設計。
condition_vec <- rep(c("voiced", "voiceless"), times = n_obs / 2)

# ---- 話者差・単語差を「先に」作っておく ----
# 実際の音声データでは、同じ条件でも話者によってVOTの平均が違い（声道長・
# 発話速度・方言）、単語によっても違う（後続母音・語長・頻度）。
# 擬似データでもこの構造を明示的に作っておかないと、第5回以降で使う
# 混合効果モデルが「推定すべきばらつきが最初から存在しない」データを
# 相手にすることになり、分散推定値がすべて0（singular fit）になってしまう。
speaker_intercept <- rnorm(n_speaker, mean = 0, sd = 6)
# 話者ごとのVOTの底上げ量。どの条件にも一律にかかる（＝ランダム切片）。
speaker_slope <- rnorm(n_speaker, mean = 0, sd = 10)
# 話者ごとの「有声と無声をどれだけ大きく作り分けるか」の差（＝ランダム傾き）。
# 話者間のばらつきは無声（VOTが長い側）で特に大きいことが知られているので、
# voiceless条件にだけ加える。
word_intercept <- rnorm(n_word, mean = 0, sd = 3)
# 単語ごとの癖（＝項目のランダム切片）。

vot_data <- tibble(
  speaker   = paste0("S", sprintf("%02d", speaker_idx)),
  word      = paste0("word_", word_idx),
  gender    = rep(rep(c("M", "F"), each = n_speaker / 2), each = n_word),
  condition = condition_vec,
  # VOTは「固定効果 + 話者差 + 単語差 + 残差」を足し合わせて作る。
  # これは第5回で当てはめる混合効果モデルの式そのものであり、
  # モデルが推定した分散をこの生成時の値と見比べられるようにしてある。
  VOT = 20 +                                              # 有声条件の基準値
    if_else(condition == "voiceless", 55, 0) +            # 無声条件の上乗せ（→約75ms）
    speaker_intercept[speaker_idx] +                      # 話者のランダム切片
    if_else(condition == "voiceless",                     # 話者のランダム傾き
            speaker_slope[speaker_idx], 0) +
    word_intercept[word_idx] +                            # 単語のランダム切片
    # 残差。無声条件のほうがばらつきが大きい（実際のVOTデータもそうなっている）。
    # 第3回のモデル診断で「等分散性が崩れている」例として使う。
    rnorm(n_obs, mean = 0, sd = if_else(condition == "voiced", 4, 8)),
  F0 = rnorm(n_obs, mean = 130, sd = 15)  # F0はここでは条件と無関係に生成
)

vot_data <- vot_data %>%
  mutate(
    # 文字列のままだとRが「順序のない値」として扱うため、
    # 話者・単語・性別・条件をすべてfactor（因子）型に変換しておく。
    # 特にconditionはlevels=でvoiced/voicelessの順序を明示し、
    # 以降のモデルの切片が「voiced群の平均」になるよう固定する。
    speaker   = factor(speaker),
    word      = factor(word),
    gender    = factor(gender),
    condition = factor(condition, levels = c("voiced", "voiceless"))
  )

glimpse(vot_data)  # 型と最初の数行を確認。数値であるべき列が<chr>のままなら要注意

# --- 実行結果 ---
# Rows: 200
# Columns: 6
# $ speaker   <fct> S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, S01, ...
# $ word      <fct> word_1, word_2, word_3, word_4, word_5, word_6, word_7, ...
# $ gender    <fct> M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, M, ...
# $ condition <fct> voiced, voiceless, voiced, voiceless, voiced, voiceless, ...
# $ VOT       <dbl> 28.129829, 88.042064, 30.742652, 94.104833, 28.438207, ...
# $ F0        <dbl> 119.0617, 144.9710, 148.8772, 148.7330, 109.2904, ...
#
# 【出力の読み方】
# ・Rows: 200 / Columns: 6 が行数と列数。10話者×20語=200行になっているかを
#   まず確認する。ここが合っていなければ、以降の分析はすべて無意味になる。
# ・各行の <fct> <dbl> が**型**である。<fct>はfactor（水準を持つカテゴリ）、
#   <dbl>は小数を含む数値。speaker・word・gender・conditionがすべて<fct>、
#   VOTとF0が<dbl>になっていれば正しい。
#   **VOTが<chr>（文字列）になっていたら、そこで止まって読み込みを見直す。**
#   文字列のままでは平均もモデルも計算できず、しかもエラーではなく
#   「変な結果」として静かに進んでしまうことがある。
# ・conditionの並びがvoiced, voiceless, voiced, voiceless...と1行おきに
#   交互になっている。これは意図した設計どおり（どの話者も両条件を10回ずつ持つ）。
# ・speakerが先頭でS01ばかり並ぶのは、話者ごとに20行がまとまっているため。
#   行の並び順自体は分析結果に影響しない（第7回のAR(1)を使うときだけ例外）。


■ この回の位置づけ：

第1回から第15回までで、音声研究で使われる分析手法をひととおり扱った。しかし手法を個別に知っていることと、自分の研究を最初から最後まで進められることは別である。実際に詰まるのは、**「自分のデータにはどれを使えばよいのか」**と**「結果をどう書けばよいのか」**の2点であることが多い。

本章は新しい手法を導入しない。代わりに、手法を選ぶための判断基準、論文に書くべき情報、そして半年後の自分が再現できる形で残す方法をまとめる。

■ 1. 手法の選び方：応答変数の型から決める：

分析手法は、研究の問いではなく**応答変数（測ったもの）の型**から決まる。以下の対応表は、本書で扱った手法をこの観点から整理したものである。

| 測ったもの | 型 | 使う手法 | 本書の該当回 |
|---|---|---|---|
| VOT・持続時間・フォルマント | 連続量 | 線形混合効果モデル | 第5回 |
| 反応時間 | 連続量（右に裾） | 対数変換して線形混合効果モデル | 第11回 |
| 正誤・有無・2択の同定 | 二値 | ロジスティックGLMM | 第6回・第10回 |
| 回数（言い直し・エラー数） | カウント | ポアソン/負の二項GLMM | 第6回 |
| 5段階・7段階の評定 | 順序 | 累積リンク混合モデル | 第11回 |
| F0やフォルマントの軌跡 | 連続量×時間 | GAMM | 第7回 |
| 複数の音響指標を同時に | 多変量 | 多変量ベイズモデル | 第9回 |
| 母音カテゴリの判別 | 分類 | 判別分析 | 第14回 |

応答変数の型が決まれば手法はほぼ一意に決まる。**迷うのはたいてい、応答変数が何かを自分でも決めていないときである。**

ランダム効果の構造は、応答変数ではなく**設計**から決まる。判断の手順は3つの問いに答えるだけでよい。(1) 同じ話者から複数の観測を取っているか → `(1 | speaker)` を入れる。(2) 同じ語・同じ刺激から複数の観測を取っているか → `(1 | word)` を入れる。(3) その予測子は、話者（あるいは語）の**内側で変化しているか** → 変化しているならランダム傾きを入れられる。変化していないなら入れられない（第5回の演習で確かめたとおり、入れても意味のない数値が返るだけである）。

■ 2. 何を報告するか：

混合効果モデルの結果を論文に書くとき、最低限必要な情報は決まっている。以下を満たさない報告は、読者が結果を評価できず、追試もできない。

In [ ]:
source_model <- lmer(VOT ~ condition + (1 + condition | speaker) + (1 | word),
                     data = vot_data)

# 固定効果を報告用の1行にまとめる関数
report_fixed <- function(model, term) {
  co <- coef(summary(model))[term, ]
  ci <- confint(model, parm = term, method = "Wald")
  sprintf("beta = %.2f, SE = %.2f, 95%% CI [%.2f, %.2f], t(%.1f) = %.2f, p = %s",
          co[["Estimate"]], co[["Std. Error"]], ci[1], ci[2],
          co[["df"]], co[["t value"]],
          format.pval(co[["Pr(>|t|)"]], digits = 2, eps = 0.001))
}

cat("【固定効果】\n")
cat(report_fixed(source_model, "conditionvoiceless"), "\n\n")

cat("【モデル式】", deparse(formula(source_model)), "\n")
cat("【データ規模】観測", nobs(source_model),
    "／話者", ngrps(source_model)["speaker"],
    "／語", ngrps(source_model)["word"], "\n\n")

cat("【ランダム効果の標準偏差】\n")
vc <- as.data.frame(VarCorr(source_model))
print(vc[, c("grp", "var1", "var2", "sdcor")])

cat("\n【環境】\n")
cat(R.version.string, "\n")
cat("lme4", as.character(packageVersion("lme4")),
    "/ lmerTest", as.character(packageVersion("lmerTest")), "\n")

# --- 実行結果 ---
# 【固定効果】
# beta = 52.50, SE = 5.56, 95% CI [41.61, 63.39], t(10.5) = 9.45, p = <0.001
#
# 【モデル式】 VOT ~ condition + (1 + condition | speaker) + (1 | word)
# 【データ規模】観測 200 ／話者 10 ／語 20
#
# 【ランダム効果の標準偏差】
#        grp               var1               var2      sdcor
# 1     word        (Intercept)               <NA>  3.4236224
# 2  speaker        (Intercept)               <NA>  4.0342320
# 3  speaker conditionvoiceless               <NA> 16.6816950
# 4  speaker        (Intercept) conditionvoiceless -0.2933389
# 5 Residual               <NA>               <NA>  5.9778352
#
# 【環境】（実行した環境によって異なる）
# R version 4.6.1 (2026-06-24)
# lme4 2.0.1 / lmerTest 3.2.1
#
# 【出力の読み方】
# ・**p値だけを書いてはいけない。** 報告に必要なのは、推定値（効果の大きさ）、
#   標準誤差または信頼区間（推定の精度）、検定統計量と自由度、そしてp値である。
#   上の1行はこれをすべて含んでいる。
# ・**信頼区間を必ず入れる。** [41.61, 63.39] という区間は、p値には無い情報を
#   持っている。効果が41msから63msの間にあり、少なくとも40msはあるらしい、
#   と読める。p < 0.001 という表記だけではこれが分からない。
#   第12回で見たとおり、有意でなかったときはなおさら区間が要る。
# ・**モデル式をそのまま書く。** 「混合効果モデルを用いた」だけでは、
#   ランダム効果に何を入れたのかが分からない。ランダム傾きの有無で
#   結論が変わりうる（第3回と第5回の比較）以上、式は結果の一部である。
# ・**データ規模は観測数だけでは足りない。** 200観測と書くと大規模に見えるが、
#   実質的な情報量は話者10名である。話者数と項目数を併記する。
# ・**ランダム効果のSDも報告する。** 話者間のばらつきが4.03msなのに対し、
#   条件の効果の話者差が16.68msもあることは、それ自体が知見である。
#   「効果は一定ではなく、話者によって大きく異なる」と書けるからである。
# ・**自由度の計算方法を明記する。** 混合効果モデルの自由度は一意に定まらず、
#   Satterthwaite近似とKenward-Roger近似で値が違う。上の t(10.5) は
#   lmerTestの既定（Satterthwaite）である。書かないと再現できない。
# ・**Rとパッケージのバージョンを書く。** 上の出力は実行環境によって変わる。
#   lme4の推定アルゴリズムは版によって変わることがあり、数値が微妙に動く。
#   `sessionInfo()` の出力を補足資料に付けるのが最も確実である。
# ・★逆に、**書かなくてよいもの**もある。有意性の星印（***）は情報量が無く、
#   多くの学術誌が推奨していない。AICやBICは、モデル比較をしたときだけ書く。


■ 3. ★よくある誤り：分析を決める前にデータを見る：

ここまでの章で扱った誤りは、いずれも手法の選択や解釈の問題だった。しかし最も結果を歪めるのは、**手続きの順序**の問題である。データを見てから分析方法を決めると、どれだけ正しい手法を使っても結果は信用できなくなる。

具体的には次のような判断が該当する。外れ値の除去基準を、結果を見てから決める。有意にならなかったので共変量を追加する。3つの音響指標を測って、有意だった1つだけを報告する。ランダム効果の構造を、収束したものではなく有意になったもので選ぶ。いずれも個々の判断としては正当化の余地があるように見えるが、**選択の自由度が結果に有利な方向へ使われている**点が共通している。

In [ ]:
# 「有意になるまで分析を変える」ことの威力を数字で確かめる。
# 効果がまったく無いデータで、3つの音響指標を測り、
# 「どれか1つでも有意なら報告する」という方針を取るとどうなるか。
set.seed(2026)

n_sim <- 2000
any_significant <- replicate(n_sim, {
  n <- 40
  g <- rep(c("a", "b"), each = n / 2)
  # 3つの指標はいずれも群と無関係（真の効果はすべてゼロ）
  p_values <- c(
    t.test(rnorm(n) ~ g)$p.value,
    t.test(rnorm(n) ~ g)$p.value,
    t.test(rnorm(n) ~ g)$p.value
  )
  any(p_values < 0.05)
})

cat("真の効果はすべてゼロ。1つの指標だけを見るなら誤検出は約5%のはず。\n")
cat("3指標のどれかが有意になる確率:", round(mean(any_significant), 3), "\n")
cat("理論値 1 - 0.95^3          :", round(1 - 0.95^3, 3), "\n")

# --- 実行結果 ---
# 真の効果はすべてゼロ。1つの指標だけを見るなら誤検出は約5%のはず。
# 3指標のどれかが有意になる確率: 0.142
# 理論値 1 - 0.95^3          : 0.143
#
# 【出力の読み方】
# ・**3つ測って1つでも有意なら報告する、という方針だけで、誤検出率が
#   5%から14.2%へ跳ね上がる。** 理論値の14.3%とほぼ一致している。
#   この14.2%には、不正も恣意的な操作も含まれていない。
#   ただ「測ったものを全部見て、有意だったものを報告した」だけである。
# ・音声研究ではVOT・F0・持続時間・フォルマントと複数の指標を同時に測ることが
#   多いので、この問題は他分野より深刻になりやすい。
#   さらに外れ値基準・変換の有無・ランダム効果構造の選択が加われば、
#   自由度は容易に数十通りに達する。
# ・★対処は多重比較の調整ではない。**分析を始める前に、何を主要な指標とし、
#   どう分析するかを書き出しておく**ことである。書いた時点でそれ以外は
#   探索的分析になり、「探索的である」と明記して報告すればよい。
#   確認的分析と探索的分析を分けて書くことは、どちらかを捨てることではない。
# ・実務的には、次の3点を録音を始める前に決めて記録しておけば足りる。
#   (1) 主要な応答変数は何か（1つか、多くても2つ）
#   (2) 外れ値の除去基準（絶対値か、平均±何SDか）
#   (3) モデル式（固定効果とランダム効果の構造、収束しなかったときの縮退手順）
#   ★とくに(3)を決めておくと、収束しないときに慌てて構造をいじり回して
#   「有意になった構造」を選んでしまう事故を防げる。


■ 4. 再現できる形で残す：

半年後に査読者から「この図をもう一度作ってほしい」と言われたとき、作り直せるだろうか。再現性は他人のためだけのものではなく、**自分のための保険**である。

In [ ]:
# 再現性のために最低限やること

# (1) 乱数を使うなら必ずseedを固定する
set.seed(42)

# (2) 分析に使った環境を記録する。この出力を補足資料に添える。
#     sessionInfo() は読み込んだパッケージのバージョンまで全部記録する。
si <- sessionInfo()
cat("R:", si$R.version$version.string, "\n")
cat("プラットフォーム:", si$platform, "\n")
cat("読み込み済みパッケージ数:", length(si$otherPkgs), "\n")
# 実際の論文では以下をそのまま貼る:
# sessionInfo()

# (3) 中間結果ではなく、生データから最終結果までの手順をコードとして残す。
#     手作業でExcelを触った工程が1つでもあると、そこで再現性は切れる。

# (4) ファイル名に日付とバージョンを埋め込まない。
#     analysis_final_v3_修正版.R のような名前は、どれが最新か分からなくなる。
#     バージョン管理（git）を使うか、少なくとも1つのファイルを更新し続ける。

cat("\n--- 再現性チェックリスト ---\n")
checklist <- c(
  "set.seed() を書いたか",
  "生データを上書きしていないか（加工は別ファイルに書き出す）",
  "手作業の工程がゼロか（Excelでの並べ替え・手入力を含む）",
  "sessionInfo() を記録したか",
  "図表を生成するコードが残っているか",
  "欠測・除外した観測の数と理由を記録したか"
)
for (i in seq_along(checklist)) cat(sprintf("  %d. %s\n", i, checklist[i]))

# 【解説】
# ・(1)のseedは、乱数を使う分析——ブートストラップ、ベイズのMCMC、
#   交差検証のデータ分割、シミュレーション——のすべてに必要である。
#   第13回の検定力計算も、seedを変えれば値が動く。
# ・(2)のsessionInfo()が重要なのは、**パッケージの更新で結果が変わりうる**
#   ためである。lme4の最適化アルゴリズムやmgcvの既定値は版によって変わる。
#   「同じコードなのに数値が違う」の原因はほぼこれである。
# ・(3)が実務上いちばん破られやすい。「とりあえずExcelで並べ替えて…」の一手が
#   入ると、その工程は誰にも再現できなくなる。**読み込みから結果まで、
#   1本のスクリプトを実行するだけで再現できる状態**を目標にする。
#   工程が増えてきたら、付録で扱う targets パッケージが助けになる。
# ・(4)について。ファイル名でのバージョン管理は必ず破綻する。
#   本書のノートブックも `_generate_notebooks.py` という1つの生成スクリプトを
#   直して再生成する方式にしてあり、.ipynb を手で編集しない。
#   **同じ内容が2か所にあると、必ず片方だけ古くなる。**


■ 演習：

1. 自分の研究（または関心のあるテーマ）について、応答変数の型を特定し、上の対応表からどの手法を使うべきかを決めよ。
2. その研究のランダム効果構造を、本章の3つの問いに答える形で書き出せ。
3. 主要な応答変数・外れ値基準・モデル式を明記した分析計画を、200字程度で書いてみよ。

■ 演習の解答：

この演習に唯一の正解は無い。以下は、本書が使ってきたVOTの研究を例にした記入例である。自分の研究に置き換えて書いてみてほしい。

**1. 応答変数の型と手法**

応答変数はVOT（ミリ秒）で、連続量である。対応表より線形混合効果モデル（第5回）を使う。ただし第3回で確認したとおり無声条件の分散が大きいので、モデル診断で等分散性が問題になるようなら対数変換を検討する。

**2. ランダム効果構造**

(1) 同じ話者から20語ずつ取っているので `(1 | speaker)` が要る。(2) 同じ語を10名が発話しているので `(1 | word)` が要る。(3) conditionは話者の内側では変化する（各話者が両条件を発話する）ので話者のランダム傾きを入れられる。一方、各語はどちらか一方の条件にしか現れないので、語のランダム傾きは入れられない。したがって `VOT ~ condition + (1 + condition | speaker) + (1 | word)` となる。

**3. 分析計画の記入例**

「主要な応答変数はVOT（ms）とする。手動でセグメンテーションし、閉鎖の解放から声帯振動開始までを測定する。0ms未満および150msを超える値は測定誤りとみなして除外し、除外数を報告する。分析には`VOT ~ condition + (1 + condition | speaker) + (1 | word)` の線形混合効果モデルを用い、自由度はSatterthwaite近似で求める。収束しない場合は、まずランダム傾きと切片の相関を外し（数値の対比を用いる）、それでも収束しなければランダム傾きを落とす。F0と持続時間も測定するが、これらの分析は探索的なものとして区別して報告する。」

この程度の分量でよい。**重要なのは長さではなく、データを見る前に書いてあることである。**